# Nexora Motors — Análisis Estadístico y Lógica de Negocio
**Autor:** María Isabel Durango Pérez  
**Dataset:** coches_clean.csv (498 registros · 19 variables)  
**Objetivo:** Extraer métricas estadísticas que alimenten la narrativa ejecutiva del dashboard

In [2]:
import pandas as pd
import numpy as np

# Carga del dataset limpio
df = pd.read_csv('../data/coches_clean.csv')

print(f"✅ Dataset cargado correctamente")
print(f"   Filas:    {df.shape[0]}")
print(f"   Columnas: {df.shape[1]}")
print()
print(df.head(3).to_string())

✅ Dataset cargado correctamente
   Filas:    498
   Columnas: 19

           marca                                    modelo  ano_vehic  antiguedad rango_antiguedad       km combustible  puertas     cv transmision ubicacion   particular  precio    segmento_precio  precio_por_cv  stats_visto  stats_favorito  stats_contactado  engagement_score
0           SEAT  LEON 1. 9 TDI 150CV SPORT FORMULA RACING     2004.0          17   Clásico (>12a)  241.665      Diesel      5.0  150.0      manual  Alicante  Profesional     3.9    Económico (<5K)          26.00         7212               8                 2            4046.0
1           AUDI                                        A6     2005.0          16   Clásico (>12a)  188.212      Diesel      5.0  140.0      manual   Almería   Particular     5.7  Accesible (5-10K)          40.71        22604              11                 0           11632.0
2  MERCEDES-BENZ                           CLASE E E 220 D     2017.0           4  Reciente (4-7a)  

In [3]:
# Variables numéricas clave para el análisis
vars_numericas = ['precio', 'km', 'cv', 'antiguedad', 'precio_por_cv', 'engagement_score']

print("=" * 60)
print("MÉTRICAS DE TENDENCIA CENTRAL Y DISPERSIÓN")
print("=" * 60)

for col in vars_numericas:
    print(f"\n📊 {col.upper()}")
    print(f"   Media:             {df[col].mean():.2f}")
    print(f"   Mediana:           {df[col].median():.2f}")
    print(f"   Moda:              {df[col].mode()[0]:.2f}")
    print(f"   Desv. típica:      {df[col].std():.2f}")
    print(f"   Varianza:          {df[col].var():.2f}")
    print(f"   Mínimo:            {df[col].min():.2f}")
    print(f"   Máximo:            {df[col].max():.2f}")
    print(f"   IQR (Q3-Q1):       {(df[col].quantile(0.75) - df[col].quantile(0.25)):.2f}")

MÉTRICAS DE TENDENCIA CENTRAL Y DISPERSIÓN

📊 PRECIO
   Media:             14.71
   Mediana:           12.50
   Moda:              13.99
   Desv. típica:      10.43
   Varianza:          108.83
   Mínimo:            1.25
   Máximo:            90.00
   IQR (Q3-Q1):       10.78

📊 KM
   Media:             98.11
   Mediana:           84.39
   Moda:              10.00
   Desv. típica:      83.82
   Varianza:          7025.06
   Mínimo:            1.00
   Máximo:            836.00
   IQR (Q3-Q1):       116.67

📊 CV
   Media:             137.58
   Mediana:           122.00
   Moda:              110.00
   Desv. típica:      58.03
   Varianza:          3367.22
   Mínimo:            60.00
   Máximo:            555.00
   IQR (Q3-Q1):       40.00

📊 ANTIGUEDAD
   Media:             6.74
   Mediana:           5.00
   Moda:              1.00
   Desv. típica:      5.46
   Varianza:          29.85
   Mínimo:            0.00
   Máximo:            27.00
   IQR (Q3-Q1):       8.75

📊 PRECIO_POR_CV
   Me

In [4]:
print("=" * 60)
print("PRECIO MEDIO POR TIPO DE COMBUSTIBLE")
print("=" * 60)

combustible_stats = df.groupby('combustible').agg(
    total_vehiculos  = ('precio', 'count'),
    precio_medio     = ('precio', 'mean'),
    precio_mediana   = ('precio', 'median'),
    km_medio         = ('km', 'mean'),
    cv_medio         = ('cv', 'mean')
).round(2)

print(combustible_stats.to_string())

print()
print("💡 CONCLUSIÓN EJECUTIVA:")
mejor = combustible_stats['precio_medio'].idxmax()
menor = combustible_stats['precio_medio'].idxmin()
print(f"   El combustible '{mejor}' concentra los vehículos más caros")
print(f"   El combustible '{menor}' representa el segmento más económico")

PRECIO MEDIO POR TIPO DE COMBUSTIBLE
             total_vehiculos  precio_medio  precio_mediana  km_medio  cv_medio
combustible                                                                   
Diesel                   272         13.20           11.50    125.31    133.76
Gasolina                 169         13.90           11.99     78.06    136.36
Híbrido                   57         24.32           20.99     27.71    159.39

💡 CONCLUSIÓN EJECUTIVA:
   El combustible 'Híbrido' concentra los vehículos más caros
   El combustible 'Diesel' representa el segmento más económico


In [5]:
print("=" * 60)
print("DISTRIBUCIÓN POR SEGMENTO DE PRECIO")
print("=" * 60)

segmento_stats = df.groupby('segmento_precio', observed=True).agg(
    total_vehiculos  = ('precio', 'count'),
    precio_medio     = ('precio', 'mean'),
    km_medio         = ('km', 'mean'),
    cv_medio         = ('cv', 'mean'),
    antiguedad_media = ('antiguedad', 'mean')
).round(2)

# Añadir porcentaje del total
segmento_stats['pct_mercado'] = (
    segmento_stats['total_vehiculos'] / len(df) * 100
).round(1)

print(segmento_stats.to_string())

print()
print("💡 CONCLUSIÓN EJECUTIVA:")
segmento_mayor = segmento_stats['total_vehiculos'].idxmax()
pct_mayor = segmento_stats.loc[segmento_mayor, 'pct_mercado']
print(f"   El segmento '{segmento_mayor}' domina el mercado")
print(f"   con el {pct_mayor}% del catálogo total de Nexora Motors")

DISTRIBUCIÓN POR SEGMENTO DE PRECIO
                   total_vehiculos  precio_medio  km_medio  cv_medio  antiguedad_media  pct_mercado
segmento_precio                                                                                    
Accesible (5-10K)              140          7.57    128.96    113.54              9.51         28.1
Económico (<5K)                 54          3.80    191.03    113.43             15.07         10.8
Lujo (>35K)                     23         46.09     66.36    266.39              4.39          4.6
Medio (10-20K)                 205         14.68     71.75    133.62              4.21         41.2
Premium (20-35K)                76         26.17     55.97    170.70              3.24         15.3

💡 CONCLUSIÓN EJECUTIVA:
   El segmento 'Medio (10-20K)' domina el mercado
   con el 41.2% del catálogo total de Nexora Motors


In [6]:
print("=" * 60)
print("TOP 10 COMUNIDADES POR VOLUMEN Y PRECIO MEDIO")
print("=" * 60)

ubicacion_stats = df.groupby('ubicacion').agg(
    total_vehiculos = ('precio', 'count'),
    precio_medio    = ('precio', 'mean'),
    precio_mediana  = ('precio', 'median'),
    km_medio        = ('km', 'mean')
).round(2)

# Ordenar por volumen de oferta
ubicacion_stats = ubicacion_stats.sort_values(
    'total_vehiculos', ascending=False
).head(10)

ubicacion_stats['pct_mercado'] = (
    ubicacion_stats['total_vehiculos'] / len(df) * 100
).round(1)

print(ubicacion_stats.to_string())

print()
print("💡 CONCLUSIÓN EJECUTIVA:")
top1 = ubicacion_stats.index[0]
top1_pct = ubicacion_stats.iloc[0]['pct_mercado']
top1_precio = ubicacion_stats.iloc[0]['precio_medio']
print(f"   '{top1}' lidera la oferta con el {top1_pct}% del mercado")
print(f"   y un precio medio de {top1_precio:.2f}K€")

TOP 10 COMUNIDADES POR VOLUMEN Y PRECIO MEDIO
            total_vehiculos  precio_medio  precio_mediana  km_medio  pct_mercado
ubicacion                                                                       
Cantabria               105         15.03           14.49     57.61         21.1
Madrid                   79         14.15           12.50     96.68         15.9
Zaragoza                 41         10.28            5.99    146.93          8.2
Pontevedra               39         15.05           14.40    149.64          7.8
Valladolid               35          6.56            5.90    157.66          7.0
Malaga                   34         23.22           21.70     76.70          6.8
Alicante                 31         24.54           18.99     94.78          6.2
Castellon                30         11.92           10.90    120.27          6.0
Las Palmas               16         10.08            9.49     59.39          3.2
Barcelona                14         13.25           10.88     6

In [7]:
print("=" * 60)
print("TOP 10 MARCAS POR VOLUMEN Y PRECIO MEDIO")
print("=" * 60)

marca_stats = df.groupby('marca').agg(
    total_vehiculos  = ('precio', 'count'),
    precio_medio     = ('precio', 'mean'),
    precio_mediana   = ('precio', 'median'),
    km_medio         = ('km', 'mean'),
    cv_medio         = ('cv', 'mean'),
    antiguedad_media = ('antiguedad', 'mean')
).round(2)

marca_stats = marca_stats.sort_values(
    'total_vehiculos', ascending=False
).head(10)

marca_stats['pct_mercado'] = (
    marca_stats['total_vehiculos'] / len(df) * 100
).round(1)

print(marca_stats.to_string())

print()
print("💡 CONCLUSIÓN EJECUTIVA:")
top_marca = marca_stats.index[0]
top_precio = marca_stats.index[marca_stats['precio_medio'].argmax()]
print(f"   '{top_marca}' es la marca con mayor volumen de oferta")
print(f"   '{top_precio}' es la marca con mayor precio medio en el top 10")

TOP 10 MARCAS POR VOLUMEN Y PRECIO MEDIO
               total_vehiculos  precio_medio  precio_mediana  km_medio  cv_medio  antiguedad_media  pct_mercado
marca                                                                                                          
OPEL                        72         14.37           14.50     46.60    119.89              3.71         14.5
PEUGEOT                     71         17.19           13.00     93.51    128.24              4.87         14.3
HYUNDAI                     43         16.37           15.90     74.78    129.42              3.74          8.6
RENAULT                     33         11.42            9.99     89.40    119.55              7.33          6.6
FORD                        33         10.33            8.40    103.87    122.18              8.67          6.6
BMW                         32         16.89           14.45    169.15    211.69             10.00          6.4
VOLKSWAGEN                  27         13.34           12.90   

In [8]:
print("=" * 60)
print("MATRIZ DE CORRELACIONES — VARIABLES CLAVE")
print("=" * 60)

vars_corr = ['precio', 'km', 'cv', 'antiguedad', 'precio_por_cv', 'engagement_score']

corr = df[vars_corr].corr().round(3)
print(corr.to_string())

print()
print("💡 CONCLUSIONES EJECUTIVAS:")
print()
print("   CORRELACIONES FUERTES (|r| > 0.5):")
for i in range(len(vars_corr)):
    for j in range(i+1, len(vars_corr)):
        r = corr.iloc[i, j]
        if abs(r) > 0.5:
            direccion = "positiva 📈" if r > 0 else "negativa 📉"
            print(f"   · {vars_corr[i]} ↔ {vars_corr[j]}: r={r} ({direccion})")

print()
print("   CORRELACIONES MODERADAS (0.3 < |r| < 0.5):")
for i in range(len(vars_corr)):
    for j in range(i+1, len(vars_corr)):
        r = corr.iloc[i, j]
        if 0.3 < abs(r) <= 0.5:
            direccion = "positiva 📈" if r > 0 else "negativa 📉"
            print(f"   · {vars_corr[i]} ↔ {vars_corr[j]}: r={r} ({direccion})")

MATRIZ DE CORRELACIONES — VARIABLES CLAVE
                  precio     km     cv  antiguedad  precio_por_cv  engagement_score
precio             1.000 -0.360  0.608      -0.471          0.753             0.044
km                -0.360  1.000  0.072       0.652         -0.517             0.128
cv                 0.608  0.072  1.000       0.088          0.024             0.057
antiguedad        -0.471  0.652  0.088       1.000         -0.668             0.006
precio_por_cv      0.753 -0.517  0.024      -0.668          1.000            -0.005
engagement_score   0.044  0.128  0.057       0.006         -0.005             1.000

💡 CONCLUSIONES EJECUTIVAS:

   CORRELACIONES FUERTES (|r| > 0.5):
   · precio ↔ cv: r=0.608 (positiva 📈)
   · precio ↔ precio_por_cv: r=0.753 (positiva 📈)
   · km ↔ antiguedad: r=0.652 (positiva 📈)
   · km ↔ precio_por_cv: r=-0.517 (negativa 📉)
   · antiguedad ↔ precio_por_cv: r=-0.668 (negativa 📉)

   CORRELACIONES MODERADAS (0.3 < |r| < 0.5):
   · precio ↔ km: r=-0

In [9]:
print("=" * 60)
print("RESUMEN EJECUTIVO — NEXORA MOTORS")
print("=" * 60)

print("""
📌 HALLAZGOS CLAVE PARA EL COMITÉ DIRECTIVO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. MERCADO
   · Precio medio del catálogo: 14.71K€ (mediana 12.50K€)
   · El segmento Medio (10-20K€) representa el 41.2% del mercado
   · El híbrido es el combustible más caro (24.32K€ de media)

2. GEOGRAFÍA
   · Cantabria lidera la oferta con el 21.1% del catálogo
   · Málaga y Alicante tienen los precios medios más altos
     del top 10 (23K€ y 24K€ respectivamente)

3. MARCAS
   · OPEL y PEUGEOT dominan el volumen (28.8% conjunto)
   · MERCEDES-BENZ tiene el precio medio más alto (24.79K€)
   · CITROEN ofrece el precio medio más bajo del top 10 (9.02K€)

4. DETERMINANTES DEL PRECIO
   · La potencia (CV) es el predictor más fuerte del precio (r=0.608)
   · A más antigüedad, menor precio (r=-0.471)
   · A más kilómetros, menor precio (r=-0.360)

5. OPORTUNIDAD DETECTADA
   · Los híbridos tienen menor kilometraje medio (27K km)
   · y mayor precio medio (24.32K€) — segmento premium en expansión
""")

print("✅ Análisis estadístico completado")
print(f"   Dataset analizado: {df.shape[0]} vehículos · {df.shape[1]} variables")

RESUMEN EJECUTIVO — NEXORA MOTORS

📌 HALLAZGOS CLAVE PARA EL COMITÉ DIRECTIVO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. MERCADO
   · Precio medio del catálogo: 14.71K€ (mediana 12.50K€)
   · El segmento Medio (10-20K€) representa el 41.2% del mercado
   · El híbrido es el combustible más caro (24.32K€ de media)

2. GEOGRAFÍA
   · Cantabria lidera la oferta con el 21.1% del catálogo
   · Málaga y Alicante tienen los precios medios más altos
     del top 10 (23K€ y 24K€ respectivamente)

3. MARCAS
   · OPEL y PEUGEOT dominan el volumen (28.8% conjunto)
   · MERCEDES-BENZ tiene el precio medio más alto (24.79K€)
   · CITROEN ofrece el precio medio más bajo del top 10 (9.02K€)

4. DETERMINANTES DEL PRECIO
   · La potencia (CV) es el predictor más fuerte del precio (r=0.608)
   · A más antigüedad, menor precio (r=-0.471)
   · A más kilómetros, menor precio (r=-0.360)

5. OPORTUNIDAD DETECTADA
   · Los híbridos tienen menor kilometraje medio (27K km)
   · y mayor precio medio (24.32K€) 